# Portable execution configuration

This public copy was prepared from the executed JPCO revision notebook.

Machine-specific absolute paths and saved execution outputs were removed.
The original numerical summaries are provided separately in this revision
package.

By default, the notebook treats its current working directory as the
revision root. If required, set the following environment variables before
starting Jupyter:

- `GW_GLITCH_REVISION_ROOT`: directory containing the revision scripts and
  result subdirectories;
- `GW_GLITCH_PROJECT_ROOT`: parent `GW_Glitch` project directory containing
  historical processed datasets;
- `GW_GLITCH_RESEARCH_ROOT`: parent research directory;
- `GW_GLITCH_IMAGE_PROJECT_ROOT`: archived Gravity Spy image-project root.

The full source datasets, trained checkpoints, prediction arrays, and local
execution logs are not stored in Git.


In [ ]:
import os
from pathlib import Path

PORTABLE_REVISION_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_REVISION_ROOT",
        Path.cwd(),
    )
).expanduser().resolve()

PORTABLE_PROJECT_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_PROJECT_ROOT",
        PORTABLE_REVISION_ROOT.parent.parent,
    )
).expanduser().resolve()

PORTABLE_RESEARCH_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_RESEARCH_ROOT",
        PORTABLE_PROJECT_ROOT.parent,
    )
).expanduser().resolve()

PORTABLE_DESKTOP_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_DESKTOP_ROOT",
        PORTABLE_RESEARCH_ROOT.parent,
    )
).expanduser().resolve()

PORTABLE_IMAGE_PROJECT_ROOT = Path(
    os.environ.get(
        "GW_GLITCH_IMAGE_PROJECT_ROOT",
        PORTABLE_RESEARCH_ROOT / "Glitch_Image",
    )
).expanduser().resolve()

print("Portable revision root:", PORTABLE_REVISION_ROOT)
print("Portable project root:", PORTABLE_PROJECT_ROOT)
print("Portable research root:", PORTABLE_RESEARCH_ROOT)
print("Portable image-project root:", PORTABLE_IMAGE_PROJECT_ROOT)


In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import sys

import numpy as np
import pandas as pd

ROOT = Path(
    str(PORTABLE_REVISION_ROOT)
)

FREEZE_DIR = (
    ROOT
    / "revision_evidence_freeze_v1"
)

OUTPUT_DIR = (
    ROOT
    / "manuscript_evidence_v1"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

INVENTORY_PATH = (
    FREEZE_DIR
    / "authoritative_evidence_inventory.csv"
)

FREEZE_PATH = (
    FREEZE_DIR
    / "revision_evidence_freeze.json"
)

print("Python:", sys.version.split()[0])
print("Executable:", sys.executable)
print("Operating system:", platform.platform())
print("Revision root:", ROOT)
print("Freeze directory:", FREEZE_DIR)
print("Output directory:", OUTPUT_DIR)

assert INVENTORY_PATH.is_file()
assert FREEZE_PATH.is_file()

print()
print("MANUSCRIPT-EVIDENCE WORKSPACE READY")

In [ ]:
def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with path.open("rb") as handle:
        while True:
            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


inventory = pd.read_csv(
    INVENTORY_PATH
)

with FREEZE_PATH.open() as handle:
    freeze_record = json.load(handle)

assert (
    freeze_record["status"]
    == "revision_evidence_frozen"
)

assert (
    freeze_record[
        "numerical_experiments_complete"
    ]
    is True
)

assert (
    freeze_record[
        "further_training_planned"
    ]
    is False
)

verified = []

for row in inventory.itertuples(
    index=False
):
    path = Path(row.path)

    exists = path.is_file()

    observed_hash = (
        sha256_file(path)
        if exists
        else None
    )

    passed = (
        exists
        and observed_hash == row.sha256
    )

    verified.append({
        "evidence_name": row.evidence_name,
        "exists": exists,
        "hash_verified": passed,
    })

verification = pd.DataFrame(
    verified
)

print(
    verification.to_string(
        index=False
    )
)

assert len(verification) == 19
assert verification["exists"].all()
assert verification["hash_verified"].all()

print()
print("ALL 19 FROZEN EVIDENCE FILES REVERIFIED")

In [ ]:
EVIDENCE = {
    row.evidence_name: Path(row.path)
    for row in inventory.itertuples(
        index=False
    )
}

print("Available evidence keys:")

for key in sorted(EVIDENCE):
    print(" ", key)

assert len(EVIDENCE) == 19

print()
print("EVIDENCE LOOKUP READY")

In [ ]:
TABLE_KEYS = [
    "frozen_split_summary",
    "xgboost_temporal_results",
    "mlp_temporal_results",
    "fttransformer_temporal_results",
    "targeted_ablation_summary",
    "permutation_baseline_summary",
    "permutation_importance_per_seed",
    "permutation_importance_summary",
    "permutation_rank_agreement",
    "complexity_summary",
    "latency_summary",
    "bandwidth_summary",
    "bandwidth_paired_differences",
]

tables = {
    key: pd.read_csv(
        EVIDENCE[key]
    )
    for key in TABLE_KEYS
}

for key, frame in tables.items():
    print("=" * 80)
    print(key)
    print("Rows:", len(frame))
    print("Columns:")
    
    for column in frame.columns:
        print(" ", column)

print()
print("ALL MANUSCRIPT-FACING TABLES LOADED")

In [ ]:
COMPACT_KEYS = [
    "frozen_split_summary",
    "xgboost_temporal_results",
    "mlp_temporal_results",
    "fttransformer_temporal_results",
    "targeted_ablation_summary",
    "permutation_rank_agreement",
    "complexity_summary",
    "latency_summary",
    "bandwidth_summary",
    "bandwidth_paired_differences",
]

for key in COMPACT_KEYS:
    frame = tables[key]

    print()
    print("=" * 100)
    print(key.upper())
    print("=" * 100)

    if len(frame) <= 50:
        print(
            frame.to_string(
                index=False
            )
        )
    else:
        print(
            frame.head(20).to_string(
                index=False
            )
        )
        print()
        print(
            f"Showing first 20 of "
            f"{len(frame)} rows."
        )

print()
print("COMPACT EVIDENCE INSPECTION COMPLETE")

In [ ]:
schema_report = {}

for key, frame in tables.items():
    schema_report[key] = {
        "path": str(EVIDENCE[key]),
        "rows": int(len(frame)),
        "columns": frame.columns.tolist(),
        "dtypes": {
            column: str(dtype)
            for column, dtype
            in frame.dtypes.items()
        },
    }

schema_path = (
    OUTPUT_DIR
    / "evidence_table_schema_report.json"
)

schema_path.write_text(
    json.dumps(
        schema_report,
        indent=2,
    )
    + "\n"
)

print(
    "SAVED:",
    schema_path,
)

print()
print("SCHEMA REPORT COMPLETE")
print("NO EXPERIMENTAL FILES WERE MODIFIED")

In [ ]:
MODEL_TABLE_KEYS = {
    "XGBoost": "xgboost_temporal_results",
    "MLP": "mlp_temporal_results",
    "FT-Transformer": (
        "fttransformer_temporal_results"
    ),
}

temporal_frames = []

for model, key in MODEL_TABLE_KEYS.items():
    frame = tables[key].copy()

    frame.insert(
        0,
        "model",
        model,
    )

    frame["split"] = (
        frame["condition"]
        .str.replace(
            "__original_nine",
            "",
            regex=False,
        )
        .str.replace(
            "__timestamp_free_six",
            "",
            regex=False,
        )
    )

    frame["feature_set"] = np.where(
        frame["condition"].str.endswith(
            "__original_nine"
        ),
        "Original nine",
        "Timestamp-free six",
    )

    temporal_frames.append(frame)

temporal_master = pd.concat(
    temporal_frames,
    ignore_index=True,
)

temporal_master = temporal_master[
    [
        "model",
        "split",
        "feature_set",
        "train_rows",
        "test_rows",
        "supported_classes",
        "seeds",
        "weighted_f1_mean",
        "weighted_f1_sd",
        "macro_f1_all24_mean",
        "macro_f1_all24_sd",
        "accuracy_mean",
        "accuracy_sd",
    ]
].sort_values(
    [
        "model",
        "split",
        "feature_set",
    ]
).reset_index(drop=True)

temporal_path = (
    OUTPUT_DIR
    / "table_temporal_generalization.csv"
)

temporal_master.to_csv(
    temporal_path,
    index=False,
)

print(
    temporal_master.to_string(
        index=False
    )
)

print()
print("SAVED:", temporal_path)

In [ ]:
weighted_lookup = (
    temporal_master
    .set_index(
        [
            "model",
            "split",
            "feature_set",
        ]
    )["weighted_f1_mean"]
)

effect_records = []

for model in MODEL_TABLE_KEYS:
    for split in [
        "random_split",
        "chronological_split",
    ]:
        original = weighted_lookup.loc[
            model,
            split,
            "Original nine",
        ]

        timestamp_free = weighted_lookup.loc[
            model,
            split,
            "Timestamp-free six",
        ]

        effect_records.append({
            "analysis": "timestamp_removal",
            "model": model,
            "feature_set": (
                "Timestamp-free minus original"
            ),
            "split": split,
            "reference_weighted_f1": original,
            "comparison_weighted_f1": (
                timestamp_free
            ),
            "difference": (
                timestamp_free - original
            ),
            "difference_percentage_points": (
                100
                * (
                    timestamp_free
                    - original
                )
            ),
        })

for model in MODEL_TABLE_KEYS:
    for feature_set in [
        "Original nine",
        "Timestamp-free six",
    ]:
        random_score = weighted_lookup.loc[
            model,
            "random_split",
            feature_set,
        ]

        chronological_score = (
            weighted_lookup.loc[
                model,
                "chronological_split",
                feature_set,
            ]
        )

        effect_records.append({
            "analysis": (
                "chronological_minus_random"
            ),
            "model": model,
            "feature_set": feature_set,
            "split": (
                "Chronological minus random"
            ),
            "reference_weighted_f1": (
                random_score
            ),
            "comparison_weighted_f1": (
                chronological_score
            ),
            "difference": (
                chronological_score
                - random_score
            ),
            "difference_percentage_points": (
                100
                * (
                    chronological_score
                    - random_score
                )
            ),
        })

temporal_effects = pd.DataFrame(
    effect_records
)

effects_path = (
    OUTPUT_DIR
    / "table_temporal_effects.csv"
)

temporal_effects.to_csv(
    effects_path,
    index=False,
)

print(
    temporal_effects.to_string(
        index=False
    )
)

print()
print("SAVED:", effects_path)

In [ ]:
REFERENCE_FEATURE_SET = {
    "remove_start_time_ns": (
        "Original nine"
    ),
    "remove_peak_time_pair": (
        "Original nine"
    ),
    "timestamp_free_remove_amplitude_snr": (
        "Timestamp-free six"
    ),
    "timestamp_free_remove_q": (
        "Timestamp-free six"
    ),
}

targeted = tables[
    "targeted_ablation_summary"
].copy()

reference_scores = []

for row in targeted.itertuples(
    index=False
):
    feature_set = REFERENCE_FEATURE_SET[
        row.condition
    ]

    reference = weighted_lookup.loc[
        row.model,
        "random_split",
        feature_set,
    ]

    reference_scores.append(reference)

targeted[
    "reference_feature_set"
] = [
    REFERENCE_FEATURE_SET[
        condition
    ]
    for condition in targeted["condition"]
]

targeted[
    "reference_weighted_f1_mean"
] = reference_scores

targeted[
    "difference_from_reference"
] = (
    targeted["test_weighted_f1_mean"]
    - targeted[
        "reference_weighted_f1_mean"
    ]
)

targeted[
    "difference_percentage_points"
] = (
    100
    * targeted[
        "difference_from_reference"
    ]
)

targeted_manuscript = targeted[
    [
        "model",
        "condition",
        "reference_feature_set",
        "seeds",
        "reference_weighted_f1_mean",
        "test_weighted_f1_mean",
        "test_weighted_f1_sample_std",
        "difference_percentage_points",
        "test_macro_f1_all24_mean",
        "test_accuracy_mean",
    ]
].sort_values(
    [
        "model",
        "condition",
    ]
).reset_index(drop=True)

targeted_path = (
    OUTPUT_DIR
    / "table_targeted_ablations.csv"
)

targeted_manuscript.to_csv(
    targeted_path,
    index=False,
)

print(
    targeted_manuscript.to_string(
        index=False
    )
)

print()
print("SAVED:", targeted_path)

In [ ]:
rank_agreement = tables[
    "permutation_rank_agreement"
].copy()

rank_agreement["seed"] = (
    rank_agreement["seed"]
    .astype(str)
)

permutation_agreement = (
    rank_agreement.loc[
        (
            rank_agreement["seed"]
            == "mean_of_seed_importances"
        )
        & (
            rank_agreement["status"]
            == "defined"
        )
    ]
    .copy()
)

assert len(
    permutation_agreement
) == 24

permutation_agreement[
    "model_pair"
] = (
    permutation_agreement["model_a"]
    + " vs "
    + permutation_agreement["model_b"]
)

permutation_agreement = (
    permutation_agreement[
        [
            "condition",
            "metric",
            "model_pair",
            "n_features",
            "spearman_rho",
        ]
    ]
    .sort_values(
        [
            "condition",
            "metric",
            "model_pair",
        ]
    )
    .reset_index(drop=True)
)

agreement_path = (
    OUTPUT_DIR
    / "table_common_permutation_agreement.csv"
)

permutation_agreement.to_csv(
    agreement_path,
    index=False,
)

print(
    permutation_agreement.to_string(
        index=False
    )
)

print()
print("SAVED:", agreement_path)

In [ ]:
agreement_ranges = (
    permutation_agreement
    .groupby(
        [
            "condition",
            "metric",
        ],
        as_index=False,
    )
    .agg(
        model_pairs=(
            "model_pair",
            "count",
        ),
        minimum_spearman_rho=(
            "spearman_rho",
            "min",
        ),
        maximum_spearman_rho=(
            "spearman_rho",
            "max",
        ),
        mean_spearman_rho=(
            "spearman_rho",
            "mean",
        ),
    )
)

agreement_ranges_path = (
    OUTPUT_DIR
    / "table_common_permutation_agreement_ranges.csv"
)

agreement_ranges.to_csv(
    agreement_ranges_path,
    index=False,
)

print(
    agreement_ranges.to_string(
        index=False
    )
)

print()
print(
    "SAVED:",
    agreement_ranges_path,
)

In [ ]:
complexity = tables[
    "complexity_summary"
].copy()

single_event_latency = (
    tables["latency_summary"]
    .loc[
        lambda frame:
        frame["batch_size"] == 1
    ]
    .copy()
)

assert len(single_event_latency) == 3

operational = complexity.merge(
    single_event_latency[
        [
            "model",
            "latency_median_ms_mean",
            "latency_median_ms_sd",
        ]
    ],
    on="model",
    how="inner",
    validate="one_to_one",
)

operational = operational[
    [
        "model",
        "training_seeds",
        "test_weighted_f1_mean",
        "test_weighted_f1_sd",
        "deployment_bundle_mib_mean",
        "deployment_bundle_mib_sd",
        "latency_median_ms_mean",
        "latency_median_ms_sd",
        "rss_loaded_model_after_data_release_mib_mean",
        "rss_loaded_model_after_data_release_mib_sd",
        "parameter_count_mean",
        "xgb_tree_count_mean",
        "xgb_total_leaves_mean",
    ]
].sort_values(
    "test_weighted_f1_mean",
    ascending=False,
).reset_index(drop=True)

operational_path = (
    OUTPUT_DIR
    / "table_operational_complexity.csv"
)

operational.to_csv(
    operational_path,
    index=False,
)

print(
    operational.to_string(
        index=False
    )
)

print()
print("SAVED:", operational_path)

In [ ]:
bandwidth = tables[
    "bandwidth_summary"
].copy()

bandwidth_differences = tables[
    "bandwidth_paired_differences"
].copy()

bandwidth_manuscript = bandwidth.merge(
    bandwidth_differences[
        [
            "model",
            "split",
            "mean_difference",
            "sample_std_difference",
            "mean_difference_percentage_points",
        ]
    ],
    on=[
        "model",
        "split",
    ],
    how="inner",
    validate="one_to_one",
)

bandwidth_manuscript = (
    bandwidth_manuscript[
        [
            "model",
            "split",
            "seeds",
            "test_weighted_f1_mean",
            "test_weighted_f1_sample_std",
            "mean_difference",
            "sample_std_difference",
            "mean_difference_percentage_points",
            "test_macro_f1_all24_mean",
            "test_accuracy_mean",
        ]
    ]
    .sort_values(
        [
            "model",
            "split",
        ]
    )
    .reset_index(drop=True)
)

bandwidth_path = (
    OUTPUT_DIR
    / "table_bandwidth_augmentation.csv"
)

bandwidth_manuscript.to_csv(
    bandwidth_path,
    index=False,
)

print(
    bandwidth_manuscript.to_string(
        index=False
    )
)

print()
print("SAVED:", bandwidth_path)

In [ ]:
LATEX_DIR = (
    OUTPUT_DIR
    / "latex_tables"
)

LATEX_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

LATEX_TABLES = {
    "temporal_generalization.tex": (
        temporal_master
    ),
    "temporal_effects.tex": (
        temporal_effects
    ),
    "targeted_ablations.tex": (
        targeted_manuscript
    ),
    "common_permutation_agreement.tex": (
        permutation_agreement
    ),
    "common_permutation_agreement_ranges.tex": (
        agreement_ranges
    ),
    "operational_complexity.tex": (
        operational
    ),
    "bandwidth_augmentation.tex": (
        bandwidth_manuscript
    ),
}

for filename, frame in LATEX_TABLES.items():
    path = LATEX_DIR / filename

    latex = frame.to_latex(
        index=False,
        float_format=lambda value: (
            f"{value:.4f}"
        ),
        na_rep="--",
        escape=True,
    )

    path.write_text(latex)

    print(
        "SAVED:",
        path,
    )

print()
print("ALL LATEX TABLE FRAGMENTS WRITTEN")

In [ ]:
def markdown_cell(value):
    if pd.isna(value):
        return "--"

    if isinstance(
        value,
        (float, np.floating),
    ):
        return f"{value:.6f}"

    if isinstance(
        value,
        (int, np.integer),
    ):
        return str(value)

    return (
        str(value)
        .replace("|", r"\|")
        .replace("\n", " ")
    )


def dataframe_to_markdown(frame):
    columns = [
        str(column)
        for column in frame.columns
    ]

    header = (
        "| "
        + " | ".join(columns)
        + " |"
    )

    separator = (
        "| "
        + " | ".join(
            ["---"] * len(columns)
        )
        + " |"
    )

    rows = []

    for values in frame.itertuples(
        index=False,
        name=None,
    ):
        rows.append(
            "| "
            + " | ".join(
                markdown_cell(value)
                for value in values
            )
            + " |"
        )

    return "\n".join(
        [
            header,
            separator,
            *rows,
        ]
    )


REPORT_TABLES = {
    "Temporal generalization": (
        temporal_master
    ),
    "Temporal and timestamp effects": (
        temporal_effects
    ),
    "Targeted feature ablations": (
        targeted_manuscript
    ),
    "Common permutation agreement": (
        permutation_agreement
    ),
    "Permutation agreement ranges": (
        agreement_ranges
    ),
    "Operational complexity": (
        operational
    ),
    "Bandwidth augmentation": (
        bandwidth_manuscript
    ),
}

report_lines = [
    "# JPCO Revision Numerical Evidence",
    "",
    (
        "Generated exclusively from the "
        "hash-verified revision evidence freeze."
    ),
    "",
]

for title, frame in REPORT_TABLES.items():
    report_lines.extend([
        f"## {title}",
        "",
        dataframe_to_markdown(frame),
        "",
    ])

report_path = (
    OUTPUT_DIR
    / "manuscript_numerical_evidence.md"
)

report_path.write_text(
    "\n".join(report_lines)
    + "\n"
)

assert report_path.is_file()
assert report_path.stat().st_size > 0

print(
    "SAVED:",
    report_path,
)

print(
    "File size:",
    f"{report_path.stat().st_size:,} bytes",
)

print()
print("NUMERICAL EVIDENCE REPORT COMPLETE")
print(
    "No additional package was installed."
)

In [ ]:
preview_lines = (
    report_path
    .read_text()
    .splitlines()
)

print(
    "\n".join(
        preview_lines[:35]
    )
)

assert (
    preview_lines[0]
    == "# JPCO Revision Numerical Evidence"
)

assert any(
    line
    == "## Temporal generalization"
    for line in preview_lines
)

assert any(
    line
    == "## Operational complexity"
    for line in preview_lines
)

print()
print("MARKDOWN REPORT INSPECTION PASSED")

In [ ]:
required_outputs = [
    OUTPUT_DIR
    / "table_temporal_generalization.csv",

    OUTPUT_DIR
    / "table_temporal_effects.csv",

    OUTPUT_DIR
    / "table_targeted_ablations.csv",

    OUTPUT_DIR
    / "table_common_permutation_agreement.csv",

    OUTPUT_DIR
    / "table_common_permutation_agreement_ranges.csv",

    OUTPUT_DIR
    / "table_operational_complexity.csv",

    OUTPUT_DIR
    / "table_bandwidth_augmentation.csv",

    OUTPUT_DIR
    / "manuscript_numerical_evidence.md",
]

for path in required_outputs:
    assert path.is_file(), (
        f"Missing output: {path}"
    )

    print(
        "READY:",
        path.name,
        f"({path.stat().st_size:,} bytes)",
    )

assert len(
    list(LATEX_DIR.glob("*.tex"))
) == 7

print()
print(
    "LaTeX fragments:",
    len(
        list(
            LATEX_DIR.glob("*.tex")
        )
    ),
)

print()
print(
    "MANUSCRIPT-EVIDENCE PACKAGE "
    "COMPLETION CHECK PASSED"
)